# 공통 설정: 임베딩 제공자 선택

원본: [`embeddings.py`](embeddings.py)

`EMBED_PROVIDER`에 따라 OpenAI 임베딩 또는 사내 REST API를 사용한다. 문서와 질문을 같은 임베딩 공간에 놓는 RAG 실습의 공통 진입점이다.

## 이번 실습 목표

`embed_documents`와 `embed_query`의 역할을 구분하고, HTTP 실패가 예외로 드러나는 지점을 찾는다.

## 실행 전 확인

원본 정의 셀은 네트워크를 호출하지 않는다. `get_embeddings()` 생성 및 `embed_query()` 호출은 제공자 설정, 자격증명, 네트워크가 준비된 뒤에만 실행한다.

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()


def _required(name: str) -> str:
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f'{name} 환경변수가 필요합니다. 실행할 환경 파일을 확인하세요.')
    return value


def get_embeddings(**kwargs):
    """
    EMBED_PROVIDER 환경변수에 따라 임베딩 인스턴스를 반환한다.
    - openai: OpenAI 임베딩 (사외)
    - samsung (기본값): 삼성 사내 임베딩 REST API
    """
    provider = os.getenv("EMBED_PROVIDER", "samsung").strip().lower()

    if provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        _required("OPENAI_API_KEY")
        if "model" not in kwargs:
            kwargs["model"] = _required("OPENAI_EMBEDDING_MODEL")
        return OpenAIEmbeddings(**kwargs)

    # ── 삼성 사내 임베딩 ──────────────────────────────────────────────────────
    # pilot/day2/04_RAG/정답/rag_retriever.py 의 MyCustomEmbeddings 참고.
    # 사내 망에서만 동작함.
    #
    elif provider == "samsung":
        import requests
        from langchain_core.embeddings import Embeddings
        from config.samsung import embedding_config

        class _SamsungEmbeddings(Embeddings):
            def __init__(self, config: dict):
                self.config = config

            def embed_documents(self, texts: list[str]) -> list[list[float]]:
                # 02_embeddings.py와 같이 한 번에 10개씩 나눠 요청한다.
                result = []
                for i in range(0, len(texts), 10):
                    resp = requests.post(
                        self.config["url"],
                        json={"input": texts[i:i + 10]},
                        headers=self.config["headers"],
                        timeout=120,
                    )
                    resp.raise_for_status()
                    result.extend(item["embedding"] for item in resp.json()["data"])
                return result

            def embed_query(self, text: str) -> list[float]:
                return self.embed_documents([text])[0]

        return _SamsungEmbeddings(embedding_config())
    # ─────────────────────────────────────────────────────────────────────────

    raise ValueError(f"지원하지 않는 EMBED_PROVIDER: {provider!r}. 'openai' 또는 'samsung'을 사용하세요.")


## CodeMate와 함께하기

CodeMate에게 제공자별 입력과 반환형을 설명하게 한다. 생성한 설명을 실제 `get_embeddings`와 `_SamsungEmbeddings` 구현에 대조한다.

## 결과 검토

문서 여러 개와 질문 하나의 반환 형태는 어떻게 다른가? 요청 헤더에 필요한 인증 필드 이름은 무엇인가? 비밀값은 노트북에 적지 않는다.